# Reference implementation — INjecting BM25 Scores


It measures **injected vs non-injected**. Comparison against the paper's published figures happens separately.

Five steps: **retrieve → format the score → build the input → score → evaluate.**

##  Configuration

Every constant is quoted from the paper. The two that matter most: GLOBAL_MAX_BM25 = 50

In [1]:
from pathlib import Path

GLOBAL_MIN_BM25 = 0
GLOBAL_MAX_BM25 = 50

# BM25 98 is is the max score

INJECT_MAX_INT = 196

RERANK_DEPTH       = 1000 
QUERY_MAX_TOKENS   = 30
PASSAGE_MAX_TOKENS = 200 
BM25_K1, BM25_B    = 0.82, 0.68 # "tuned parameters from the Anserini documentation"

COLLECTIONS = {
    "TREC DL'19":  dict(irds="msmarco-passage/trec-dl-2019/judged", topics=43,   graded=True),
    "TREC DL'20":  dict(irds="msmarco-passage/trec-dl-2020/judged", topics=54,   graded=True),
    "MSMARCO DEV": dict(irds="msmarco-passage/dev/small",           topics=6980, graded=False),
}


# ── our checkpoints ───────────────────────────────────────────────────────────────────
# Trained under the paper cross-entropy loss, early stopping on DL'20 nDCG@10,
# Eq.3 ordering (score BETWEEN query and passage). Identical data, seed and schedule;
# the two differ only in whether the score is injected.
MODELS = {
    "CAT":     dict(repo="Amdestya/ce-cat-minilm-l12",     inject=None),
    "BM25CAT": dict(repo="Amdestya/ce-bm25cat-minilm-l12", inject="score_middle"),
}

DATA = Path("./paper_tables_data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./paper_tables_out");  OUT.mkdir(exist_ok=True)


## 1a · Setup

Cache paths, imports, and which collection to run. The index is ~3.9 GB and the HuggingFace cache another ~3.2 GB, so both are redirected out of your home directory.

In [2]:
"""Step 1a — caches, imports, and the collection to run."""
import os, json, urllib.request
import pandas as pd

# ~7 GB lands here in total. Override with IR_CACHE_ROOT if this partition is small.
CACHE = Path(os.environ.get("IR_CACHE_ROOT", "./ircache")).resolve()
os.environ.setdefault("PYTERRIER_HOME", str(CACHE / "pyterrier"))
os.environ.setdefault("HF_HOME",        str(CACHE / "hf"))
(CACHE / "pyterrier").mkdir(parents=True, exist_ok=True)
(CACHE / "hf").mkdir(parents=True, exist_ok=True)

# These must be set BEFORE pyterrier is imported -- it reads PYTERRIER_HOME at import time.
import pyterrier as pt

COLLECTION = "TREC DL'19"      # one at a time. DEV is 6,980 queries -> ~2 h per model.
cfg = COLLECTIONS[COLLECTION]

print(f"pyterrier {pt.__version__}")
print(f"cache     {CACHE}")
print(f"running   {COLLECTION}  ({cfg['topics']} topics, graded={cfg['graded']})")

pyterrier 0.13.1
cache     /root/ircache
running   TREC DL'19  (43 topics, graded=True)


## 1b · The index

A prebuilt Terrier index over all 8.8M MS MARCO passages — ~3.2 GB download, ~3.9 GB on disk. Downloaded once, then cached; its own sha256 is verified on the way in.

In [3]:
"""Step 1b — the prebuilt MS MARCO index.

Separate cell because this is the slow, failure-prone step. PyTerrier verifies the artifact's
sha256 as it streams, so a corrupted download raises rather than handing you a subtly broken index.
If that happens, just re-run this cell.
"""
index = pt.Artifact.from_hf("pyterrier/msmarco-passage.terrier")
print(index)

https://huggingface.co/datasets/pyterrier/msmarco-passage.terrier/resolve/main/artifact.tar.lz4:   0%|          | 472k/2.94G [00:00<48:01, 1.10MB/s]    

extracting data.direct.bf [486.0 MB]


https://huggingface.co/datasets/pyterrier/msmarco-passage.terrier/resolve/main/artifact.tar.lz4:  17%|█▋        | 497M/2.94G [00:10<00:25, 105MB/s]  

extracting data.document.fsarrayfile [177.1 MB]


https://huggingface.co/datasets/pyterrier/msmarco-passage.terrier/resolve/main/artifact.tar.lz4:  20%|█▉        | 592M/2.94G [00:12<00:35, 70.8MB/s]

...

extracting data.meta.idx [67.5 MB]


https://huggingface.co/datasets/pyterrier/msmarco-passage.terrier/resolve/main/artifact.tar.lz4:  37%|███▋      | 1.09G/2.94G [00:19<00:15, 128MB/s]

extracting data.meta.zdata [1.9 GB]


https://huggingface.co/datasets/pyterrier/msmarco-passage.terrier/resolve/main/artifact.tar.lz4: 100%|██████████| 2.94G/2.94G [01:16<00:00, 41.4MB/s]

extracting data.properties [4.3 KB]
extracting pt_meta.json [79 B]
TerrierIndex('/root/ircache/pyterrier/artifacts/5fb9ed05ee653302a044f774b2effcc1e67b60d84a466b82901422fdbea7346c')


## 1c · Topics and qrels

The `/judged` variant only. DL'19 ships 200 topics but just 43 carry relevance judgements — evaluating all 200 averages in zeros and silently divides every metric by ~4.

In [4]:
"""Step 1c — topics and relevance judgements."""
ds     = pt.get_dataset("irds:" + cfg["irds"])
topics = ds.get_topics()      # columns: qid, query
qrels  = ds.get_qrels()       # columns: qid, docno, label

# tripwire: if the wrong variant ever gets loaded, fail here rather than in the final table
assert len(topics) == cfg["topics"], f"expected {cfg['topics']} topics, got {len(topics)}"

print(f"{COLLECTION}: {len(topics)} topics, {len(qrels)} judgements")
print(f"relevance levels present: {sorted(qrels['label'].unique())}")
print(topics.head(3).to_string(index=False))

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/ircache/pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependencies.jar: 100%|██████████| 99.6M/99.6M [00:00<00:00, 222MB/s] 


Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/ircache/pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar: 100%|██████████| 36.6k/36.6k [00:00<00:00, 3.23MB/s]

Done


TREC DL'19: 43 topics, 9260 judgements
relevance levels present: [0, 1, 2, 3]
    qid                                 query
 156493                      do goldfish grow
1110199             what is wifi vs bluetooth
1063750 why did the us volunterilay enter ww1


Java started (triggered by _pt_tokeniser) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## 1d · Retrieve

BM25 at the paper's parameters and depth. The `score` column it returns is the raw BM25 value — normally discarded after ranking, but here it is the feature step 2 formats.

In [5]:
"""Step 1d — BM25 retrieval, keeping the raw score."""
# k1/b  depth 1000 
bm25 = index.bm25(k1=BM25_K1, b=BM25_B, num_results=RERANK_DEPTH)

first_stage = bm25.transform(topics)     # columns: qid, docno, score, rank, query

print(f"{len(first_stage):,} candidates, {len(first_stage)/len(topics):.0f} per query")
print(first_stage.head(3)[["qid", "docno", "rank", "score"]].to_string(index=False))

s = first_stage["score"]
print(f"\nraw BM25: min {s.min():.2f} | max {s.max():.2f} | mean {s.mean():.2f} | sd {s.std():.2f}")
print("paper §3.3 reports its own data as {min 0, max 98, mean 7, sd 5}")
print("  -> the MEAN is the number to check. Their DL'19 file gives 6.94, so ours should be close.")
print("  -> the MAX will be lower: 98 came from ~40M training pairs, this is 43 queries.")

23:51:49.774 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1.9 GiB of memory would be required.
42,005 candidates, 977 per query
   qid   docno  rank     score
156493 6139386     0 28.013180
156493 8182161     1 27.837165
156493 3288600     2 27.714462

raw BM25: min 7.00 | max 64.99 | mean 18.13 | sd 6.09
paper §3.3 reports its own data as {min 0, max 98, mean 7, sd 5}
  -> the MEAN is the number to check. Their DL'19 file gives 6.94, so ours should be close.
  -> the MAX will be lower: 98 came from ~40M training pairs, this is 43 queries.


## 2a · Normalise the score

`int(raw / 50 * 100)` — Min-Max in the **global** setting, then truncate. The 50 is a fixed constant from §3.3, *not* this query's maximum.

In [6]:
"""Step 2a — turn the raw BM25 float into the integer the model was trained on."""

def normalise(raw: float) -> int:
    return int(((raw - GLOBAL_MIN_BM25) / (GLOBAL_MAX_BM25 - GLOBAL_MIN_BM25)) * 100)

first_stage["inject"] = first_stage["score"].map(normalise)

# worked example, so the arithmetic is checkable by hand
r = first_stage.iloc[0]
print(f"qid {r.qid}  docno {r.docno}")
print(f"  raw BM25        {r.score}")
print(f"  / {GLOBAL_MAX_BM25} * 100   {r.score / GLOBAL_MAX_BM25 * 100}")
print(f"  truncated       {r.inject}")
print(f"  injected as     \"{r.inject}\"  (a string, not a number)")

print(first_stage.head(5)[["qid", "docno", "score", "inject"]].to_string(index=False))

qid 156493  docno 6139386
  raw BM25        28.013180289253768
  / 50 * 100   56.026360578507536
  truncated       56
  injected as     "56"  (a string, not a number)
   qid   docno     score  inject
156493 6139386 28.013180      56
156493 8182161 27.837165      55
156493 3288600 27.714462      55
156493 3288596 27.592916      55
156493 2411918 27.516750      55


## 2b · Why an integer

the integer numbers are already included in the BERT tokenizer's vocabulary, allowing for appropriate tokenization

In [7]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("microsoft/MiniLM-L12-H384-uncased")

multi = [i for i in range(INJECT_MAX_INT + 1)
         if len(tok.encode(str(i), add_special_tokens=False)) != 1]
print(f"integers 0..{INJECT_MAX_INT}: {len(multi)} are not single tokens "
      f"{'-- OK' if not multi else multi[:10]}")

# the alternative the paper rules out, and which a reproduction can easily fall into
for form in ("14", "0.14", "0.14000", "14.4"):
    ids = tok.encode(form, add_special_tokens=False)
    print(f"  {form:9s} -> {len(ids)} token(s)  {tok.convert_ids_to_tokens(ids)}")

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

integers 0..196: 0 are not single tokens -- OK
  14        -> 1 token(s)  ['14']
  0.14      -> 3 token(s)  ['0', '.', '14']
  0.14000   -> 4 token(s)  ['0', '.', '1400', '##0']
  14.4      -> 3 token(s)  ['14', '.', '4']


## 2c · Do the values land where the model expects?

Training spanned 0–196. If our injected values fall outside that, the model is reading tokens it never saw in this position.

In [8]:
"""Step 2c — check the injected values against the range the model was trained on."""
vals = first_stage["inject"]
over = (vals > INJECT_MAX_INT).sum()

print(f"injected integers: {vals.min()} .. {vals.max()}   mean {vals.mean():.1f}")
print(f"trained range was 0 .. {INJECT_MAX_INT}   -> {over} value(s) above it ({over/len(vals):.2%})")

print("\ndistribution:")
for lo in range(0, 200, 25):
    frac = ((vals >= lo) & (vals < lo + 25)).mean()
    print(f"  {lo:3d}-{lo+24:3d}  {frac:6.1%}  {'#' * int(frac * 50)}")

if over:
    print("\n  WARNING: values above the trained range. Those tokens exist in the vocabulary but")
    print("  the model has never seen them here. The paper does not clip, so neither do we -- record it.")
else:
    print("\n  all inside the trained range")

injected integers: 14 .. 129   mean 35.8
trained range was 0 .. 196   -> 0 value(s) above it (0.00%)

distribution:
    0- 24   10.9%  #####
   25- 49   78.5%  #######################################
   50- 74    9.2%  ####
   75- 99    1.1%  
  100-124    0.2%  
  125-149    0.0%  
  150-174    0.0%  
  175-199    0.0%  

  all inside the trained range


## 3a · Attach the passage text

Retrieval returns document *ids*. The cross-encoder needs the text, and this index stores it

In [9]:
get_text = pt.text.get_text(index, "text")
candidates = get_text(first_stage)


print(f"{len(candidates):,} candidates with text")
print(f"passage length: mean {candidates['text'].str.split().str.len().mean():.0f} words "
      f"(paper §4 gives the collection average as 73.1)")
print(f"\nqid    {candidates.iloc[0]['qid']}")
print(f"query  {candidates.iloc[0]['query']}")
print(f"inject {candidates.iloc[0]['inject']}")
print(f"text   {candidates.iloc[0]['text'][:110]}...")

42,005 candidates with text
passage length: mean 57 words (paper §4 gives the collection average as 73.1)

qid    156493
query  do goldfish grow
inject 56
text   A: The conditions goldfish are kept in plus their diet determine how large they will grow. I have seen goldfis...


## 3b · Build the pair

A cross-encoder tokenises `(text_a, text_b)` as `[CLS] a [SEP] b [SEP]` — no third slot, so the score is spliced into `text_a`. §4 caps the query at 30 tokens and the passage at 200.

In [10]:
def cap(text: str, max_tokens: int) -> str:
    # only touch text that exceeds the cap: decode(encode(x)) is not always x
    ids = tok.encode(text, add_special_tokens=False)
    return text if len(ids) <= max_tokens else tok.decode(ids[:max_tokens], skip_special_tokens=True)

def build_pair(inject, query, passage, score):
    q, p = cap(query, QUERY_MAX_TOKENS), cap(passage, PASSAGE_MAX_TOKENS)
    if inject is None:
        return q, p                          # CE_CAT       [CLS] q [SEP] p [SEP]
    if inject == "score_middle":
        return f"{q} [SEP] {score}", p       # paper Eq.3   [CLS] q [SEP] s [SEP] p [SEP]
    return f"{score} [SEP] {q}", p           # released     [CLS] s [SEP] q [SEP] p [SEP]

# how often does truncation actually fire?
n_q = sum(len(tok.encode(q, add_special_tokens=False)) > QUERY_MAX_TOKENS
          for q in candidates["query"].unique())

          
sample = candidates["text"].sample(min(2000, len(candidates)), random_state=42)
n_p = sum(len(tok.encode(t, add_special_tokens=False)) > PASSAGE_MAX_TOKENS for t in sample)
print(f"queries over {QUERY_MAX_TOKENS} tokens : {n_q} of {candidates['query'].nunique()}")
print(f"passages over {PASSAGE_MAX_TOKENS} tokens: ~{n_p/len(sample):.1%} (of {len(sample)} sampled)")

queries over 30 tokens : 0 of 43
passages over 200 tokens: ~0.2% (of 2000 sampled)


## 3c · Check the tokens, don't trust the string

A literal `[SEP]` in an f-string either resolves to the special token or it doesn't, and nothing downstream would tell you.

In [11]:
Q = "what is a cat"
P = "a cat is a small domesticated carnivorous mammal"
S = 44

print(f"{'arrangement':22s} {'[SEP]':>5s}  tokens")


for label, inject in [("CAT (no injection)", None),
                      ("Eq.3  score_middle", "score_middle"),
                      ("code  score_first", "score_first")]:
    a, b = build_pair(inject, Q, P, S)
    toks = tok.convert_ids_to_tokens(tok(a, b)["input_ids"])
    print(f"{label:22s} {toks.count(tok.sep_token):>5d}  {' '.join(toks[:15])}")

cat = tok.convert_ids_to_tokens(tok(*build_pair(None, Q, P, S))["input_ids"])
mid = tok.convert_ids_to_tokens(tok(*build_pair("score_middle", Q, P, S))["input_ids"])
fst = tok.convert_ids_to_tokens(tok(*build_pair("score_first",  Q, P, S))["input_ids"])



arrangement            [SEP]  tokens
CAT (no injection)         2  [CLS] what is a cat [SEP] a cat is a small domestic ##ated car ##nivorous
Eq.3  score_middle         3  [CLS] what is a cat [SEP] 44 [SEP] a cat is a small domestic ##ated
code  score_first          3  [CLS] 44 [SEP] what is a cat [SEP] a cat is a small domestic ##ated


## 4 · Cross-encoder inference

Score every pair, take the logit. Outputs are raw logits (unbounded, often negative) — only the ordering within a query matters.

In [12]:
"""Step 4 — score every pair with each checkpoint."""
import time, numpy as np, torch
from sentence_transformers import CrossEncoder

BATCH = 128
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

def score_arm(arm: str) -> np.ndarray:
    """Score all candidates with one arm, feeding it the arrangement it was TRAINED with."""
    spec = MODELS[arm]
    cache = OUT / f"scores_{arm}_{COLLECTION.replace(' ', '').replace(chr(39), '')}.npy"
    if cache.exists():
        print(f"  {arm:8s} cached  {cache.name}")
        return np.load(cache)

    model = CrossEncoder(spec["repo"], max_length=256, device=device)

    # sanity: the checkpoint must be the 1-logit regression head we expect
    n_labels = model.model.config.num_labels
    assert n_labels == 1, f"{spec['repo']} has num_labels={n_labels}, expected 1"

    pairs = [build_pair(spec["inject"], q, t, s) for q, t, s in
             zip(candidates["query"], candidates["text"], candidates["inject"])]

    # show what this arm is actually being fed -- cheap, and catches a wrong mapping
    a0, b0 = pairs[0]
    print(f"  {arm:8s} inject={str(spec['inject']):12s} text_a={a0[:60]!r}")

    t0 = time.perf_counter()
    scores = model.predict(pairs, batch_size=BATCH, show_progress_bar=True, convert_to_numpy=True)
    dt = time.perf_counter() - t0
    print(f"  {arm:8s} {len(pairs)} pairs in {dt:.0f}s ({len(pairs)/dt:.0f} pairs/s), "
          f"logits {scores.min():.2f}..{scores.max():.2f}")

    np.save(cache, scores)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return scores

scored = {arm: score_arm(arm) for arm in ("CAT", "BM25CAT")}

# ── VERIFY: the two arms must not be producing identical scores ────────────────────────
same = np.allclose(scored["CAT"], scored["BM25CAT"])
corr = np.corrcoef(scored["CAT"], scored["BM25CAT"])[0, 1]
print(f"\nCAT vs BM25CAT: identical={same}  pearson r={corr:.4f}")


device: cuda


config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

  BM25CAT  inject=score_middle text_a='do goldfish grow [SEP] 56'


Batches:   0%|          | 0/329 [00:00<?, ?it/s]

  BM25CAT  42005 pairs in 33s (1259 pairs/s), logits 0.00..1.00

CAT vs BM25CAT: identical=False  pearson r=0.8556


## 5a · Re-sort into runs

Group by query, sort descending on the new score, re-rank. The BM25 run is built the same way from the first-stage score, so all three are directly comparable.

In [13]:
from pyterrier.measures import nDCG, AP, RR

def to_run(scores):
    df = candidates[["qid", "docno", "query"]].copy()
    df["score"] = scores
    df = df.sort_values(["qid", "score"], ascending=[True, False])
    return pt.model.add_ranks(df.reset_index(drop=True))

runs = {"BM25":           to_run(candidates["score"].values),
        "MiniLM_CAT":     to_run(scored["CAT"]),
        "MiniLM_BM25CAT": to_run(scored["BM25CAT"])}

# how much did reranking actually move things?
for name, df in runs.items():
    top1 = df[df["rank"] == 0].set_index("qid")["docno"]
    if name == "BM25":
        bm25_top1 = top1
    changed = (top1 != bm25_top1.reindex(top1.index)).mean()
    print(f"{name:16s} {len(df):,} rows   top-1 differs from BM25 for {changed:.0%} of queries")

BM25             42,005 rows   top-1 differs from BM25 for 0% of queries
MiniLM_CAT       42,005 rows   top-1 differs from BM25 for 93% of queries
MiniLM_BM25CAT   42,005 rows   top-1 differs from BM25 for 95% of queries


## 5b · Write TREC run files

`qid Q0 docno rank score tag` — the standard format, so `trec_eval` or any other tool can check these independently.

In [14]:
tag = COLLECTION.replace(" ", "").replace("'", "")

for name, df in runs.items():
    p = OUT / f"{name}_{tag}.trec"
    df.assign(Q0="Q0", run=name)[["qid", "Q0", "docno", "rank", "score", "run"]] \
      .to_csv(p, sep=" ", header=False, index=False)
    print(f"  {p.name}  ({len(df):,} lines)")

print(f"\nfirst lines of {name}_{tag}.trec:")
print((OUT / f"{name}_{tag}.trec").read_text().splitlines()[0])

  BM25_TRECDL19.trec  (42,005 lines)
  MiniLM_CAT_TRECDL19.trec  (42,005 lines)
  MiniLM_BM25CAT_TRECDL19.trec  (42,005 lines)

first lines of MiniLM_BM25CAT_TRECDL19.trec:
1037798 Q0 8760866 0 0.9944852 MiniLM_BM25CAT


## 5c · Evaluate

Two experiments. Everything vs BM25 shows the pipeline works

In [15]:
# nDCG uses the graded labels directly. MAP needs a binary split, so on TREC DL we report
# BOTH thresholds; on DEV only level 1 exists, so rel>=2 would select nothing.
if cfg["graded"]:
    measures = [nDCG @ 10, AP(rel=2) @ 1000, AP(rel=1) @ 1000]
else:
    measures = [nDCG @ 10, AP(rel=1) @ 1000, RR(rel=1) @ 10]

names = list(runs)

print(f"{COLLECTION} — all systems vs BM25")
vs_bm25 = pt.Experiment(list(runs.values()), topics, qrels, eval_metrics=measures,
                        names=names, baseline=0, correction="bonferroni", round=4)
print(vs_bm25.to_string(index=False))

print(f"\n\nTHE CLAIM — BM25CAT vs its matched CAT baseline")
claim = pt.Experiment([runs["MiniLM_CAT"], runs["MiniLM_BM25CAT"]], topics, qrels,
                      eval_metrics=measures, names=["MiniLM_CAT", "MiniLM_BM25CAT"],
                      baseline=0, correction="bonferroni", round=4)
print(claim.to_string(index=False))

TREC DL'19 — all systems vs BM25
          name  nDCG@10  AP(rel=2)@1000  AP@1000  nDCG@10 +  nDCG@10 -  nDCG@10 p-value  nDCG@10 reject  nDCG@10 p-value corrected  AP(rel=2)@1000 +  AP(rel=2)@1000 -  AP(rel=2)@1000 p-value  AP(rel=2)@1000 reject  AP(rel=2)@1000 p-value corrected  AP@1000 +  AP@1000 -  AP@1000 p-value  AP@1000 reject  AP@1000 p-value corrected
          BM25   0.5048          0.2997   0.3859        NaN        NaN              NaN           False                        NaN               NaN               NaN                     NaN                  False                               NaN        NaN        NaN              NaN           False                        NaN
    MiniLM_CAT   0.6884          0.4568   0.4792       36.0        7.0         0.000010            True                   0.000020              36.0               6.0                0.000001                   True                          0.000003       33.0       10.0         0.000352            True     

## 5d · The table

Table 2's shape, our numbers. The paper marks † where BM25CAT significantly beats its matched CAT row — same convention here, from the paired test above.

In [16]:
def metric_cols(df):
    """the value columns, not the significance ones pt.Experiment adds"""
    skip = ("p-value", "reject", " +", " -")
    return [c for c in df.columns
            if c != "name" and not any(t in str(c) for t in skip)]

cols = metric_cols(vs_bm25)
cat_row = claim[claim["name"] == "MiniLM_CAT"].iloc[0]
inj_row = claim[claim["name"] == "MiniLM_BM25CAT"].iloc[0]

def dagger(col):
    """† only if BM25CAT is both higher AND significant after correction"""
    p = next((c for c in claim.columns
              if str(c).startswith(str(col)) and "p-value corrected" in str(c)), None)
    return "†" if (p is not None and inj_row[p] < 0.05
                   and inj_row[col] > cat_row[col]) else ""

print(f"Table 2 shape — {COLLECTION}")
print(f"{'Model':18s}" + "".join(f"{str(c):>20s}" for c in cols))
rows = []
for name in names:
    r = vs_bm25[vs_bm25["name"] == name].iloc[0]
    marks = [dagger(c) if name == "MiniLM_BM25CAT" else "" for c in cols]
    print(f"{name:18s}" + "".join(f"{r[c]:.4f}{m}".rjust(20) for c, m in zip(cols, marks)))
    rows.append(dict(collection=COLLECTION, system=name,
                     **{str(c): round(float(r[c]), 4) for c in cols}))

print("\n† = significantly better than MiniLM_CAT (paired t-test, Bonferroni, p<0.05)")

# the claim, stated plainly
nd = cols[0]
p_nd = next((c for c in claim.columns
             if str(c).startswith(str(nd)) and "p-value corrected" in str(c)), None)
delta = inj_row[nd] - cat_row[nd]
print(f"\nBM25CAT − CAT = {delta:+.4f} {nd}"
      + (f"   (corrected p = {inj_row[p_nd]:.4f})" if p_nd is not None else ""))

import pandas as pd
pd.DataFrame(rows).to_csv(OUT / f"table_{tag}.csv", index=False)
vs_bm25.to_csv(OUT / f"experiment_vs_bm25_{tag}.csv", index=False)
claim.to_csv(OUT / f"experiment_claim_{tag}.csv", index=False)
print(f"\nwrote {OUT}/table_{tag}.csv (+ both experiment frames)")

Table 2 shape — TREC DL'19
Model                          nDCG@10      AP(rel=2)@1000             AP@1000
BM25                            0.5048              0.2997              0.3859
MiniLM_CAT                      0.6884              0.4568              0.4792
MiniLM_BM25CAT                  0.6785              0.4424              0.4678

† = significantly better than MiniLM_CAT (paired t-test, Bonferroni, p<0.05)

BM25CAT − CAT = -0.0099 nDCG@10   (corrected p = 0.3091)

wrote paper_tables_out/table_TRECDL19.csv (+ both experiment frames)
